In [0]:
!ls /Volumes/syntheaproject/default/syntheavolume

In [0]:
%sql
create database Bronze;
use Bronze;

In [0]:
%sql
show schemas;

In [0]:
files = ['allergies.csv', 'encounters.csv', 'observations.csv', 'payer_transitions.csv', 'careplans.csv', 'imaging_studies.csv', 'organizations.csv', 'procedures.csv', 'conditions.csv', 'immunizations.csv', 'patients.csv', 'providers.csv', 'devices.csv', 'medications.csv', 'payers.csv', 'supplies.csv']

for file in files:
    df = spark.read.csv(f'/Volumes/syntheaproject/default/syntheavolume/{file}', header=True, inferSchema=True)
    
    # creating two extra columns ingestion time for efficient increment loading and source to trace any future error
    from pyspark.sql.functions import current_timestamp, lit
    df = df.withColumns({'ingested_At': current_timestamp(), 'source': lit(file) })

    # creating tables for every file so it can be efficiently used in SQL
    df.write.saveAsTable(f'syntheaproject.bronze.{file[:-4]}')


In [0]:
# for file in files:
#     spark.sql(f'Drop Table {file[:-4]}')

In [0]:
%sql
use bronze;
-- show tables;

In [0]:
%sql
select * from encounters limit 10

In [0]:
%sql
select count(*) from observations where PATIENT='f0f3bc8d-ef38-49ce-a2bd-dfdda982b271'

In [0]:
%sql
select count(*) from conditions where PATIENT='f0f3bc8d-ef38-49ce-a2bd-dfdda982b271'

In [0]:
files = ['allergies.csv', 'encounters.csv', 'observations.csv', 'payer_transitions.csv', 'careplans.csv', 'imaging_studies.csv', 'organizations.csv', 'procedures.csv', 'conditions.csv', 'immunizations.csv', 'patients.csv', 'providers.csv', 'devices.csv', 'medications.csv', 'payers.csv', 'supplies.csv']
for i in files:
    columns = []
    col = spark.catalog.listColumns(i[:-4])
    for c in col:
        columns.append(c.name)
    print(f'{i[:-4]} - {columns}')
    print()

In [0]:
%sql
select count(distinct(*)) from (select distinct(cast(CODE as String)) from observations where code in (select distinct(cast(code as string)) from conditions where code in (select distinct(cast(code as string)) from medications)))

In [0]:
%sql
use bronze;
select * from procedures limit 10

In [0]:
%sql
select count(*) from payer_transitions;

In [0]:
%sql
select distinct(encounter) from observations limit 5;

In [0]:
%sql
select * from observations where ENCOUNTER='36324342-7b0f-45e7-bcb4-629f957c65ca'

In [0]:
%sql
use bronze;

In [0]:
%sql
select count(distinct(cast(m.CODE as string))) from syntheaproject.bronze.observations o inner join syntheaproject.bronze.medications m on cast(o.CODE as string)=cast(m.CODE as string);

In [0]:
%sql
SELECT COUNT(DISTINCT(CAST(o.CODE AS STRING))) 
FROM syntheaproject.bronze.observations o 
INNER JOIN syntheaproject.bronze.procedures p 
  ON CAST(o.code AS STRING) = CAST(p.code AS STRING);

In [0]:
%sql
select count(*), count(distinct(encounter)) from allergies

In [0]:
%sql
select count(distinct(*)) from medications where ENCOUNTER is null;

In [0]:
%sql
select count(distinct(patient)) from encounters;

In [0]:
%sql
select count(distinct(patient)) from conditions;

In [0]:
%sql
select * from conditions limit 5

In [0]:
%sql
Select distinct(Encounter) from observations limit 5

In [0]:
%sql
select * from observations where ENCOUNTER='1c2a0b33-b70d-41d7-8950-27fb6f50a4b1'

In [0]:
%sql
Select * from medications where ENCOUNTER='1c2a0b33-b70d-41d7-8950-27fb6f50a4b1'

In [0]:
%sql
Select * from procedures where ENCOUNTER = '1c2a0b33-b70d-41d7-8950-27fb6f50a4b1'

In [0]:
%sql
select * from conditions limit 5

In [0]:
%sql
select * from payers

In [0]:
%sql
select * from encounters where payer='7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a'

In [0]:
%sql
select * from Encounters limit 5

In [0]:
%sql
select * from payer_transitions limit 5

In [0]:
%sql
select * from devices where ENCOUNTER='821e57ac-9304-46a9-9f9b-83daf60e9e43'

In [0]:
%sql
select * from careplans limit 5

In [0]:
%sql
select * from immunizations limit 5

In [0]:
%sql
select * from devices limit 5; 


In [0]:

# Drop ADDRESS, CITY, STATE, ZIP, LAT, LON, UTILIZATION from Provider as these are already in Organizations./ Or these columns can be removed from both organisation and provider both and a new table for adress can be created with foreign key of organisation id and via that providers and organisation both address can be fetched when needed.
# Drop Base_encounter_cost and Organization from Encounters as its always equal to Total_claim_cost and organisation can be found from provider


In [0]:
%sql
use bronze;

In [0]:
%sql
select * from organizations LIMIT 5

In [0]:
tbs = ['Patients', 'Providers', 'Organizations', 'Payers', 'Encounters', 'Medications', 'Immunizations', 'Procedures', 'Conditions', 'Allergies', 'Observations', 'Supplies', 'Devices', 'Careplans', 'Imaging_Studies', 'Payer_Transitions']
for i in tbs:
    print(f'{i}')
    spark.sql(f"describe syntheaproject.bronze.bronze_destination_{i.lower()}").show()

In [0]:

%sql
describe syntheaproject.bronze.patients

In [0]:
tbs = ['Patients', 'Providers', 'Organizations', 'Payers', 'Encounters', 'Medications', 'Immunizations', 'Procedures', 'Conditions', 'Allergies', 'Observations', 'Supplies', 'Devices', 'Careplans', 'Imaging_Studies', 'Payer_Transitions']

for i in tbs:
    spark.sql(f"describe syntheaproject.bronze.{i}")

In [0]:

%sql
use bronze;
select * from observations where cast(value as float) is not null;

In [0]:
%sql
use bronze;
select * from  patients limit 5;

In [0]:
%sql
describe history syntheaproject.bronze.bronze_destination_patients

In [0]:
%sql
show tables in syntheaproject.silver_manual

In [0]:
%sql
use bronze;
select count(*) from encounters e join medications m on e.code = m.code;